# ಪಾಠ 03 - ಏಜೆಂಟಿಕ್ ವಿನ್ಯಾಸ ಮಾದರಿಗಳು

ಈ ಪಾಠದಲ್ಲಿ, ಪರಿಣಾಮಕಾರಿಯಾಗಿ AI ಏಜೆಂಟುಗಳನ್ನು ನಿರ್ಮಿಸಲು ಮೂರು ಮೂಲಭೂತ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ಪರಿಶೀಲಿಸುತ್ತೇವೆ:

1. **ಸ್ಪಷ್ಟ ಏಜೆಂಟ್ ಸೂಚನೆಗಳು** — ಏಜೆಂಟ್ ವರ್ತನೆಗೆ ಮಾರ್ಗದರ್ಶನ ನೀಡುವ ನಿಖರ, ಪಾತ್ರ ನಿರ್ಧಾರ ಪ್ರಾಂಪ್ಟ್ಗಳನ್ನು ರಚಿಸುವುದು
2. **ಪಿಡಾಂಟಿಕ್ ಮಾದರಿಗಳೊಂದಿಗೆ ರಚಿತ output** — ಏಜೆಂಟುಗಳು ನಿರೀಕ್ಷಿತ, ಮಾನ್ಯತೆಯಾದ ಡೇಟಾವನ್ನು ಹಿಂತಿರುಗಿಸುವುದನ್ನು ಖಾತ್ರಿಪಡಿಸುವುದು
3. **ಒಂದೇ ಹೊಣೆಗಾರಿಕೆಯ ಏಜೆಂಟುಗಳು** — ಪ್ರತಿಯೊಂದು ಏಜೆಂಟ್ ಒಳ್ಳೆಯದಾಗಿ ಒಂದು ಕೆಲಸವನ್ನು ಮಾಡುವಂತೆ ವಿನ್ಯಾಸಗೊಳಿಸುವುದು

ನಾವು ಪ್ರತಿಯೊಬ್ಬ ಮಾದರಿಯನ್ನು **ಪ್ರಯಾಣ ಗಮ್ಯಸ್ಥಾನ ಶಿಫಾರಸು ಮಾಡಿಸುವ** ದೃಶ್ಯದಲ್ಲಿ ಅನ್ವಯಿಸಿ, ಗಮ್ಯಸ್ಥಾನಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುವುದು, ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುವುದು ಮತ್ತು ಲಾಜಿಸ್ಟಿಕ್ಸ್ ನಿರ್ವಹಿಸುವ ವ್ಯವಸ್ಥೆಯನ್ನು ಹಂತ ಹಂತವಾಗಿ ನಿರ್ಮಿಸುವುದನ್ನು ಕಲಿಯೋಣ.


## ಸೆಟಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಮಾದರಿ 1: ಏಜೆಂಟ್ ಸೂಚನೆಗಳನ್ನು ಸ್ಪಷ್ಟವಾಗಿ ಬರೆವುದು

가장 ಪರಿಣಾಮಕಾರಿಯಾದ ಮಾದರಿಯೂ ಅತ್ಯಂತ ಸರಳವಾದದ್ದು: ನಿಮ್ಮ ಏಜೆಂಟ್‌ಗೆ ಸ್ಪಷ್ಟ ಮತ್ತು ವಿವರವಾದ ಸೂಚನೆಗಳನ್ನು ಬರೆಯುವುದು.

ಒಳ್ಳೆಯ ಸೂಚನೆಗಳು ನಿರ್ಧರಿಸುವುದು:
- **ಯಾರು** ಏಜೆಂಟ್ ಆಗಿದ್ದಾರೆ (ವ್ಯಕ್ತಿತ್ವ ಮತ್ತು ಶೈಲಿ)
- **ಏನು** ಮಾಡಬೇಕು (ಹಂತ-ಹಂತ ಜವಾಬ್ದಾರಿಗಳು)
- **ಹೆಗೆ** ವರ್ತಿಸಬೇಕು (ನಿಬಂಧನೆಗಳು ಮತ್ತು ಶೈಲಿ)

ಕೆಳಗಿದೆ ನಾವು ಒಂದು ಪ್ರವಾಸಿ ಸಹಾಯಕ ಏಜೆಂಟ್ ಅನ್ನು ಸೃಷ್ಟಿಸುತ್ತೇವೆ, ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳೊಂದಿಗೆ ಅದು ಪ್ರತಿ ಉತ್ತರವನ್ನು ರೂಪಿಸುತ್ತದೆ.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## ಮಾದರಿ 2: ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳೊಂದಿಗೆ ನಿರ್ಮಿತ 输出

ವಿನಾಮೂಲಕ ಪಠ್ಯ ಸಂಭಾಷಣೆಗೆ ಉಪಯುಕ್ತವಾಗಿರುತ್ತದೆ, ಆದರೆ ಕೆಳಗೆ ಇರುವ ವ್ಯವಸ್ಥೆಗಳು ರಚಿಸಲಾದ ಡೇಟಾವನ್ನು ಅಗತ್ಯವಿರುತ್ತವೆ.
**ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳು** ಅನ್ನು **ಟೂಲ್ ಫಂಕ್ಷನ್**ೊಂದಿಗೆ ಜೋಡಿಸುವ ಮೂಲಕ, ನಾವು:

- ಏಜೆಂಟ್ ಔಟ್ಪುಟ್ ಗೆ ನಿಖರವಾದ ಸ್ಕೀಮಾವನ್ನು ನಿರ್ಧರಿಸಬಹುದು
- ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಪರಿಶೀಲಿಸಬಹುದು
- ಏಜೆಂಟ್ ಫಲಿತಾಂಶಗಳನ್ನು ಅಪ್ಲಿಕೇಶನ್ ಲಾಜಿಕ್ನಲ್ಲಿ ವಿಶ್ವಾಸನೀಯವಾಗಿ ಅಳವಡಿಸಬಹುದು

ಜಾರಿಗೆ ಮುಖ್ಯ ವಿಷಯವು ಏಜೆಂಟ್ ಚಾಲನಾ ಸಮಯದಲ್ಲಿ `response_format` ಅನ್ನು ಪಾಸ್ ಮಾಡುವುದಾಗಿದೆ. ಇದು ಮಾದರಿಯನ್ನು
`[response.value]` ನಲ್ಲಿ ಲಭ್ಯವಿರುವ ಪ್ರಮಾಣೀಕೃತ `TravelRecommendations` ವಸ್ತುವನ್ನು ಪರಿಗಣಿಸಲು ಬಾಧ್ಯಗೊಳಿಸುತ್ತದೆ
ವಿನಾಮೂಲಕ ಪಠ್ಯದ ಬದಲು. `get_destination_details` ಟೂಲ್ ಕೂಡ ಟೈಪ್ಡ್ `DestinationRecommendation` ಅನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತದೆ,
ಆದ್ದರಿಂದ ಡೇಟಾ ಮುಗಿಸಲು ಮುಕ್ತಾಯವರೆಗೂ ರಚನೆಗೊಂಡಿರುತ್ತದೆಯೆದು.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## ಮಾದರಿ 3: ಏಕ ಜವಾಬ್ದಾರಿ ಏಜೆಂಟರು

ಸಂಕೀರ್ಣ ಕೆಲಸಗಳಿಗೆ, ಪ್ರತಿಯೊಬ್ಬರೂ ಒಂದು ಜವಾಬ್ದಾರಿಯನ್ನು ಹೊಂದಿರುವ ಅನೇಕ ತೀವ್ರ ಏಜೆಂಟ್‌ಗಳಿಗೆ ಕಾರ್ಯವನ್ನು ಹಂಚಿಕೊಳ್ಳುವುದು ಲಾಭಕಾರಿ:

- **ಗಮ್ಯಸ್ಥಳ ತಜ್ಞ** - ಸ್ಥಳಗಳು ಮತ್ತು ಲಭ್ಯತೆ ಬಗ್ಗೆ ತಿಳಿವಳಿಕೆ ಹೊಂದಿರುವವನು
- **ಲಾಜಿಸ್ಟಿಕ್ಸ್ ಯೋಜಕ** - ವಿಮಾನೆ, ಹೋಟೆಲ್ಗಳು, ಮತ್ತು ಪ್ರವಾಸ ಹಾದಿಗಳನ್ನು ನಿರ್ವಹಿಸುವವನು

ಇದು ಸોફ್ಟ್‌ವೇರ್ ಎಂಜಿನಿಯರಿಂಗ್ ನ *ವಿಷಯ ವಿಭಾಜನೆ* ತತ್ತ್ವವನ್ನು ಅನುಸರಿಸುತ್ತದೆ — ಪ್ರತಿ ಏಜೆಂಟ್ ಅನ್ನು ಸ್ವತಂತ್ರವಾಗಿ ಪರೀಕ್ಷಿಸುವುದು, ನಿರ್ವಹಿಸುವುದು ಮತ್ತು ಸುಧಾರಿಸುವುದು ಸುಲಭವಾಗುತ್ತದೆ.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನಾವು ಪ್ರಯಾಣ ಶಿಫಾರಸ್ಸು ಮಾಡುವ ದೃಶ್ಯಕ್ಕೆ ಮೂರು ಏಜೆಂಟಿಕ್ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ಅನ್ವಯಿಸಿದ್ದೇವೆ:

| ಮಾದರಿ | ಮುಖ್ಯ ಕಲ್ಪನೆ | ಲಾಭ |
|---|---|---|
| **ಸ್ಪಷ್ಟ ನಿರ್ದೇಶನಗಳು** | ವ್ಯಕ್ತಿತ್ವ, ಜವಾಬ್ದಾರಿಗಳು, ಮತ್ತು ನಿರ್ಬಂಧಗಳನ್ನು ಮುಂಚಿತವಾಗಿ ಸಂಜ್ಞಾಪಿಸು | ಸಧೃಢ, ಬ್ರಾಂಡ್ ಆಧಾರಿತ ಏಜೆಂಟ್ ವರ್ತನೆ |
| **ರಚಿಸಲಾಗಿರುವ ಉತ್ಪನ್ನ** | ಪ್ರತಿಕ್ರಿಯೆ ಸ್ವರೂಪವಾಗಿ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಬಳಸು | ಮಾನ್ಯಗೊಳಿಸಲ್ಪಟ್ಟ, ಯಂತ್ರ ಓದಲು ಸಾಧ್ಯವಾದ ಫಲಿತಾಂಶಗಳು |
| **ಒಂದು ಜವಾಬ್ದಾರಿ** | ಪ್ರತಿ ಏಜೆಂಟ್ ಗೆ ಒಂದು ಕೇಂದ್ರಿತ ಕೆಲಸ ನೀಡಿ | ಪರೀಕ್ಷೆ ಮಾಡುವುದು, ನಿರ್ವಹಿಸುವುದು ಮತ್ತು ಸಂಯೋಜಿಸುವುದು ಸುಲಭ |

ಈ ಮಾದರಿಗಳು ಸ್ವಾಭಾವಿಕವಾಗಿ ಸಂಕಲನವಾಗುತ್ತವೆ — ನೀವು ಸ್ಪಷ್ಟ ನಿರ್ದೇಶನಗಳನ್ನು ರಚಿಸಲಾಗಿರುವ ಉತ್ಪನ್ನದೊಂದಿಗೆ ಒಟ್ಟಿಗೆ ಒಂದು ಜವಾಬ್ದಾರಿ ಏಜೆಂಟ್ ಒಳಗೆ ಸಂಯೋಜಿಸಿ ಸ್ಥಿತಿಗತ, ಉತ್ಪಾದನಾ-ಸಿದ್ಧ ವ್ಯವಸ್ಥೆಗಳನ್ನು ರಚಿಸಬಹುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
